# Retrain the thyroid image model on TN5000 (official split)

Trains EfficientNet-B0 to classify thyroid ultrasound images as **benign** or **malignant**.

* Train on TN5000's official **train** split (3,500 images), pick the best epoch on **val** (500), score the untouched **test** split (1,000) once.
* Class-weighted loss (the data is ~70% malignant) and ImageNet normalisation.
* Outputs: `cancer_efficientnet.pth`, `cancer_efficientnet.json` (preprocessing + metrics), `metrics.json`, `roc.png`, `confusion_matrix.png`, `history.csv`.

**Before you run:** `Runtime → Change runtime type → T4 GPU`. The notebook downloads TN5000 (~200 MB) from figshare itself and checks the file; Google Drive is only used to save the trained model.

Dataset (CC BY): Zhang et al., *TN5000: An Ultrasound Image Dataset for Thyroid Nodule Detection and Classification*, Scientific Data 12, 1437 (2025).

In [ ]:
# 1. Download TN5000 from figshare and verify it (about 1 minute)
import hashlib, os, subprocess
URL = 'https://figshare.com/ndownloader/files/52597376?private_link=cb6a67f17c04b29e7edd'
ZIP = '/content/TN5000.zip'
SHA256 = 'f513017e2a2c43c1c04e28d1f45e9f89ba6121c18edbedbb2482da7a57f58f88'

if not os.path.exists(ZIP):
    subprocess.run(['wget', '-q', '-O', ZIP, URL], check=True)
h = hashlib.sha256(open(ZIP, 'rb').read()).hexdigest()
print(f'{os.path.getsize(ZIP)/1e6:.1f} MB, checksum', 'OK' if h == SHA256 else f'MISMATCH ({h})')
assert h == SHA256, 'Download incomplete or changed: delete /content/TN5000.zip and re-run this cell'

# Google Drive is only used to save the trained model at the end
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Get the project code
!rm -rf /content/thyroid-ml-system
!git clone -q https://github.com/lonewolf1621/thyroid-ml-system /content/thyroid-ml-system
%cd /content/thyroid-ml-system
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (enable a GPU runtime!)')

In [ ]:
# 3. Train (about 10-15 minutes on a T4 GPU)
!python src/train_image_model.py --data /content/TN5000.zip --out /content/outputs --epochs 15 --batch-size 32

In [ ]:
# 4. Results on the held-out test set
import json
from IPython.display import Image, display
m = json.load(open('/content/outputs/metrics.json'))['test']
for k in ['accuracy','balanced_accuracy','sensitivity','specificity','precision','f1','auc','majority_class_baseline']:
    print(f'{k:>24}: {m[k]:.3f}')
print('confusion matrix:', m['confusion_matrix'])
display(Image('/content/outputs/roc.png'))
display(Image('/content/outputs/confusion_matrix.png'))

In [ ]:
# 5. Save everything to Google Drive (folder: My Drive/thyroid_model_v2)
!mkdir -p /content/drive/MyDrive/thyroid_model_v2
!cp /content/outputs/* /content/drive/MyDrive/thyroid_model_v2/
!ls -lh /content/drive/MyDrive/thyroid_model_v2